In [2]:
import pandas as pd

df = pd.read_csv('/content/creditcard.csv')

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
#check dataset
df['Class'].value_counts()

,count
Class,
0,284315
1,492


In [4]:
#create target and features
X = df.drop(columns='Class')
Y = df['Class']

In [5]:
#split before balancing
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    stratify=Y,
    random_state=42
)

In [6]:
print("Training set:")
print(Y_train.value_counts())

print("\nTest set:")
print(Y_test.value_counts())

Training set:
Class
0    227451
1       394
Name: count, dtype: int64

Test set:
Class
0    56864
1       98
Name: count, dtype: int64


In [7]:
#scaling the features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
#training the model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight='balanced',
    max_iter=2000
)

model.fit(X_train_scaled, Y_train)

print("Model trained successfully")

Model trained successfully


In [10]:
#prediction

X_test_prediction = model.predict(X_test_scaled)

In [11]:
#accuracy

from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(Y_test, X_test_prediction)

print("Test Accuracy:", test_accuracy)

#sccuracy alone doesnt matter here

Test Accuracy: 0.9755275446789088


In [12]:
#confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(Y_test, X_test_prediction)

print(cm)

[[55478  1386]
 [    8    90]]


In [13]:
#report
from sklearn.metrics import classification_report

print(classification_report(Y_test, X_test_prediction))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [14]:
#roc-auc
Y_test_probability = model.predict_proba(X_test_scaled)[:, 1]

In [15]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(Y_test, Y_test_probability)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.9720834996210077


In [16]:
#pr-auc
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(Y_test, Y_test_probability)

print("PR-AUC:", pr_auc)

PR-AUC: 0.7189705771419241


In [27]:
import joblib

joblib.dump(model, 'fraud_detection_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Model and scaler saved successfully")

Model and scaler saved successfully


In [28]:
import os

print(os.listdir())

['.config', 'creditcard.csv', '.ipynb_checkpoints', 'fraud_detection_model.pkl', 'scaler.pkl', 'archive.zip', 'sample_data']


In [31]:
sample_transaction = X_test.iloc[0].values.reshape(1, -1)

sample_transaction_scaled = loaded_scaler.transform(sample_transaction)

prediction = loaded_model.predict(sample_transaction_scaled)

print("Prediction:", prediction[0])

Prediction: 0


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [32]:
probability = loaded_model.predict_proba(sample_transaction_scaled)[0][1]

print("Fraud probability:", probability)

Fraud probability: 0.005690975919110427
